# Campaña bancaria: ingesta y diagnóstico inicial

Carolina Machado · Data science 1 · Preentrega 2

Fuente: https://archive.ics.uci.edu/dataset/222/bank+marketing

La pregunta del proyecto es si se puede estimar la contratación de un depósito antes del contacto. En esta etapa se inspecciona el dataset; todavía no se entrena un modelo.


## 1. Ingesta y primeras filas

Descargo el ZIP original de UCI y leo bank-full.csv directamente. El archivo usa punto y coma como separador.


In [ ]:
import io
import zipfile
from urllib.request import urlopen
import pandas as pd

url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
with urlopen(url, timeout=60) as response:
    contenido = response.read()
with zipfile.ZipFile(io.BytesIO(contenido)) as paquete:
    with zipfile.ZipFile(io.BytesIO(paquete.read("bank.zip"))) as subpaquete:
        df = pd.read_csv(io.BytesIO(subpaquete.read("bank-full.csv")), sep=";")

assert df.shape == (45211, 17), "Revisar la versión y lectura del CSV"
display(df.head(5).style.set_caption("Primeros cinco registros"))


## 2. Radiografía técnica

Reviso dimensiones, tipos y estadística descriptiva. Las columnas categóricas llegan como texto y las numéricas como enteros.


In [ ]:
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print("Tipos de datos:")
print(df.dtypes.to_string())
print("Información general:")
df.info()
display(df.describe().T)


### Observaciones del resumen

1. pdays tiene mediana -1 y 36.954 registros con ese valor. Según la fuente, indica ausencia de contacto en campaña anterior. Su promedio no representa una demora típica sin separar esos registros.
2. balance va de -8.019 a 102.127, mientras su mediana es 448. Hay una dispersión marcada y valores extremos que requieren inspección. Un saldo negativo puede ser válido.

La duración se conoce después de llamar y no entrará en un modelo previo al contacto.


In [ ]:
print("Registros con pdays = -1:", (df["pdays"] == -1).sum())
print("Distribución de la etiqueta:")
print(df["y"].value_counts().to_string())
print("Porcentaje de suscripciones:", round(df["y"].eq("yes").mean() * 100, 2))


## 3. Diagnóstico de faltantes

El ranking muestra el porcentaje exacto de nulos técnicos por columna, en orden descendente. Un 0 % no descarta categorías como unknown ni códigos especiales.


In [ ]:
porcentaje_nulos = (
    df.isna().mean().mul(100)
      .sort_values(ascending=False)
      .rename("porcentaje_nulos")
      .to_frame()
)
display(porcentaje_nulos.style.format({"porcentaje_nulos": "{:.2f}%"}))
print("Valores textuales unknown:")
for columna in df.select_dtypes(include="object").columns:
    cantidad = df[columna].eq("unknown").sum()
    if cantidad:
        print(f"{columna}: {cantidad:,} ({cantidad / len(df) * 100:.2f}%)")


## 4. Funciones de transformación

Distingo los contactos previos por recencia con una función formal. Normalizo la ocupación mediante una función lambda; conservo las columnas originales para comparar.


In [ ]:
def categoria_contacto_previo(dias):
    """Agrupa la recencia de un contacto anterior."""
    if dias == -1:
        return "sin contacto previo"
    elif dias <= 90:
        return "contacto reciente"
    else:
        return "contacto lejano"

df["contacto_previo_grupo"] = df["pdays"].apply(categoria_contacto_previo)
df["job_normalizado"] = df["job"].apply(lambda valor: valor.strip().lower())
display(df[["pdays", "contacto_previo_grupo", "job", "job_normalizado"]].head(5))
print(df["contacto_previo_grupo"].value_counts().to_string())
assert df["contacto_previo_grupo"].notna().all()
assert df["job_normalizado"].notna().all()


## Próximo paso

En la siguiente etapa revisaré unknown, valores extremos y variables con posible fuga de información. Para un modelo anterior a la llamada excluiré duration y separaré entrenamiento y prueba antes de ajustar transformaciones.
